# Findex Data Exploration: Unbanked Population in India and Bangladesh

This notebook explores the 2025 Global Findex database to analyze the unbanked population in India and Bangladesh. We will calculate the weighted fraction of unbanked adults and visualize the results, including a gender breakdown.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Set plot style
sns.set_theme(style="whitegrid")

## 1. Data Loading and Preprocessing

We load the Findex microdata from the Stata file. We will then filter the dataset to include only India (`IND`) and Bangladesh (`BGD`).

In [ ]:
# Load the dataset
file_path = 'Findex/findex_microdata_2025_labelled_update112425.dta'
try:
    # Using iterator=True to handle large files efficiently if needed, though here we'll just read it all for these two countries if possible, 
    # but read_stata doesn't support filtering on load easily. 
    # Given the task description implies analyzing specific countries, we'll load and then filter.
    # We use convert_categoricals=False to ensure we get numeric codes for 'account' (0/1) instead of labels ("No"/"Yes").
    df = pd.read_stata(file_path, convert_categoricals=False)
    print("Data loaded successfully.")
except FileNotFoundError:
    print(f"File not found at {file_path}. Please check the path.")

# Filter for India and Bangladesh
target_economies = ['IND', 'BGD']
df_filtered = df[df['economycode'].isin(target_economies)].copy()

print(f"Filtered data shape: {df_filtered.shape}")
print(f"Economies included: {df_filtered['economy'].unique()}")

## 2. Defining Unbanked Population

We define "unbanked" as adults who do not have an account (`account == 0`). We will create a new binary column `is_unbanked` for this purpose.

In [ ]:
# Check the 'account' variable
# 1 = Has account, 0 = No account
print("Unique values in 'account' column:", df_filtered['account'].unique())

df_filtered['is_unbanked'] = (df_filtered['account'] == 0).astype(int)

# Check value counts to verify
print("Unbanked counts (unweighted):")
print(df_filtered.groupby('economy')['is_unbanked'].value_counts())

## 3. Weighted Analysis

The survey data includes weights (`wgt`) to ensure national representation. We must use these weights when calculating fractions.

In [ ]:
def calculate_weighted_unbanked(data, group_col):
    results = []
    for group, group_df in data.groupby(group_col):
        total_weight = group_df['wgt'].sum()
        unbanked_weight = group_df[group_df['is_unbanked'] == 1]['wgt'].sum()
        fraction_unbanked = unbanked_weight / total_weight
        results.append({'Group': group, 'Unbanked_Fraction': fraction_unbanked})
    return pd.DataFrame(results)

# Overall weighted unbanked fraction by economy
overall_unbanked = calculate_weighted_unbanked(df_filtered, 'economy')
print("Weighted Fraction of Unbanked Adults:")
print(overall_unbanked)

## 4. Visualizations

### 4.1 Overall Unbanked Population

In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(x='Group', y='Unbanked_Fraction', data=overall_unbanked, palette='viridis')
plt.title('Fraction of Unbanked Adults (Weighted)', fontsize=16)
plt.xlabel('Economy', fontsize=12)
plt.ylabel('Fraction Unbanked', fontsize=12)
plt.ylim(0, 1) # Fraction is between 0 and 1
for index, row in overall_unbanked.iterrows():
    plt.text(index, row.Unbanked_Fraction + 0.02, f"{row.Unbanked_Fraction:.2%}", color='black', ha="center")
plt.show()

### 4.2 Gender Breakdown

We will now break down the unbanked population by gender. The `female` variable is coded as 1 for female and 2 for male.

In [ ]:
# Map female variable to labels for better plotting
# Assuming 1=Female, 2=Male based on standard Findex coding (needs verification from codebook if available, but standard is usually this or 0/1)
# Let's verify with the codebook info provided in the prompt: "V8 female: Respondent is female" with categories 1=Female, 2=Male.
df_filtered['Gender'] = df_filtered['female'].map({1: 'Female', 2: 'Male'})

# Calculate weighted unbanked by economy AND gender
gender_results = []
for (economy, gender), group_df in df_filtered.groupby(['economy', 'Gender']):
    total_weight = group_df['wgt'].sum()
    unbanked_weight = group_df[group_df['is_unbanked'] == 1]['wgt'].sum()
    fraction_unbanked = unbanked_weight / total_weight
    gender_results.append({'Economy': economy, 'Gender': gender, 'Unbanked_Fraction': fraction_unbanked})

gender_unbanked_df = pd.DataFrame(gender_results)
print(gender_unbanked_df)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='Economy', y='Unbanked_Fraction', hue='Gender', data=gender_unbanked_df, palette='muted')
plt.title('Fraction of Unbanked Adults by Gender (Weighted)', fontsize=16)
plt.xlabel('Economy', fontsize=12)
plt.ylabel('Fraction Unbanked', fontsize=12)
plt.ylim(0, 1)
plt.legend(title='Gender')

# Add data labels
for p in plt.gca().patches:
    if p.get_height() > 0: # Avoid labeling empty bars if any
        plt.gca().text(p.get_x() + p.get_width()/2., p.get_height() + 0.01, 
                f"{p.get_height():.2%}", 
                fontsize=10, color='black', ha='center', va='bottom')

plt.show()

## 5. Conclusions

Based on the analysis above, we can observe the comparative levels of financial exclusion in India and Bangladesh, as well as the gender gaps in account ownership.